# Lab 9: Tree models, sparsity, neural networks, SGD and chess

**Due: Wednesday 2 December 2026, before class**

**How to submit on Canvas.** Restart the kernel and Run All. Every check cell must print `ok`. Save the notebook with its outputs and upload the `.ipynb` file to the Canvas assignment for this lab.

**What a check says.** `ok  name` means your value matches the right one (to 6 decimals, or within the stated tolerance). An error that starts with `not yet` means the value is wrong or is still `None`. Fix the exercise cell and run it again. A check never shows the right value.

**Data.** The first cell downloads `diamonds.csv`, `circle.csv` and `SaratogaHouses.csv` into `data/`. Allowed libraries: `numpy`, `pandas`, `scipy`, `scikit-learn`, `torch`.

| Exercise | Task | Online lab |
|---|---|---|
| 1 | Best split by SSE | [Tree models](https://vsokolov.org/labs/tree-models-lab.html) |
| 2 | Information gain, twenty questions with chess pieces | [Tree models](https://vsokolov.org/labs/tree-models-lab.html) |
| 3 | Pruning strength chosen on validation data | [Tree models](https://vsokolov.org/labs/tree-models-lab.html) |
| 4 | Random forest and boosting scores | [Tree models](https://vsokolov.org/labs/tree-models-lab.html) |
| 5 | Why averaging trees has a floor | [Tree models](https://vsokolov.org/labs/tree-models-lab.html) |
| 6 | Lasso as a function of a z-score | [Sparsity](https://vsokolov.org/labs/sparsity-lab.html) |
| 7 | Needles in a haystack, p much larger than n | [Sparsity](https://vsokolov.org/labs/sparsity-lab.html) |
| 8 | Forward pass of a small network | [Neural nets](https://vsokolov.org/labs/neural-net-lab.html) |
| 9 | Three ReLU folds | [Neural nets](https://vsokolov.org/labs/neural-net-lab.html) |
| 10 | Softmax, cross-entropy and its gradient | [Neural nets](https://vsokolov.org/labs/neural-net-lab.html) |
| 11 | Gradient descent steps | [SGD](https://vsokolov.org/labs/sgd-lab.html) |
| 12 | The step-size ceiling | [SGD](https://vsokolov.org/labs/sgd-lab.html) |
| 13 | Constant steps hover, shrinking steps converge | [SGD](https://vsokolov.org/labs/sgd-lab.html) |
| 14 | Minibatch SGD in PyTorch | [SGD](https://vsokolov.org/labs/sgd-lab.html) |
| 15 | The chess network, value and search | [Chess](https://vsokolov.org/labs/chess-lab.html) |


In [ ]:
import copy
from hashlib import sha256
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA = Path("data")
DATA.mkdir(exist_ok=True)
BASE = "https://vsokolov.org/courses/files/600/data"
for name in ("diamonds.csv", "circle.csv", "SaratogaHouses.csv"):
    if not (DATA / name).exists():
        urlretrieve(f"{BASE}/{name}", DATA / name)

PREFIX = "aii600-trees-nn-sgd"


def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: {name} is still None")
    try:
        x = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} must be a plain number, not {type(value).__name__}")
    s = f"{round(x, ndigits):.{ndigits}f}"
    got = sha256(f"{PREFIX}|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} is not right (yours: {x:.6g})"
    print("ok ", name)


def check_array(name, value, expected, shape, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: {name} is still None")
    arr = np.asarray(value, dtype=float)
    if arr.shape != tuple(shape):
        raise AssertionError(f"not yet: {name} has shape {arr.shape}, the task asks for {tuple(shape)}")
    key = (np.round(arr, ndigits) + 0.0).tobytes()
    got = sha256(f"{PREFIX}|{name}|".encode() + key).hexdigest()[:16]
    assert got == expected, f"not yet: {name} has the right shape but some values are wrong"
    print("ok ", name)


def check_near(name, value, target, tol, relative=False):
    # for library results that can differ a little between versions
    if value is None:
        raise AssertionError(f"not yet: {name} is still None")
    gap = abs(float(value) - target) / (abs(target) if relative else 1.0)
    assert gap < tol, f"not yet: {name} is not within the tolerance of the right value (yours: {float(value):.6g})"
    print("ok ", name)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


## Part 1: Tree models

Online lab: [Tree models lab](https://vsokolov.org/labs/tree-models-lab.html).


### Exercise 1: best split by SSE

**Task.** Find the best split of the 8 diamonds in `small`. Candidate cuts are the midpoints between neighbouring distinct carat values. Diamonds with carat below the cut go left, the others go right.

**Inputs.** `small` (DataFrame with `carat` in carats and `price` in US dollars) and `sse(values)` (sum of squared deviations from the mean).

**Formula.** score(cut) = SSE(left prices) + SSE(right prices). Keep the cut with the smallest score.

**Define.** `best_cut` (carats) and `best_sse` (squared dollars). Do not round.


In [ ]:
small = pd.DataFrame([
    (0.4, 855),
    (0.99, 5893),
    (0.51, 1839),
    (0.7, 1066),
    (1.0, 4042),
    (1.0, 5544),
    (0.82, 2165),
    (0.7, 2657)
], columns=["carat", "price"])

def sse(values):
    values = np.asarray(values, float)
    return float(((values - values.mean()) ** 2).sum())

print(small.T.to_string())


In [ ]:
# Exercise 1: define best_cut, best_sse
best_cut = None
best_sse = None


In [ ]:
check("best_cut", best_cut, "a9f6470127ee27fc")
check("best_sse", best_sse, "2562b2dc05cc1c3a")


In [ ]:
# Plot: the best split. Each leaf predicts the mean price of its diamonds.
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(small.carat, small.price, color="C0", zorder=3)
left, right = small.price[small.carat < best_cut], small.price[small.carat >= best_cut]
ax.axvline(best_cut, color="k", ls="--", label=f"best cut {best_cut:.3f} carat")
ax.hlines(left.mean(), small.carat.min(), best_cut, color="C1", lw=3, label="leaf means")
ax.hlines(right.mean(), best_cut, small.carat.max(), color="C1", lw=3)
ax.set_xlabel("carat"); ax.set_ylabel("price (US dollars)"); ax.legend(frameon=False)
plt.show()


### Exercise 2: information gain, twenty questions with chess pieces

This exercise introduces entropy, a second way (besides SSE) to score a split.

- Entropy: uncertainty about the class in a node, in bits. $H = -\sum_k p_k \log_2 p_k$, where $p_k$ is the share of class $k$.
- One bit: the uncertainty of one fair yes-or-no question. Two equally likely classes: $H = -2 \cdot \frac12 \log_2 \frac12 = 1$.
- $m$ equally likely classes: $H = \log_2 m$. One class only (a pure node): $H = 0$.
- Information gain of a question: entropy before minus the size-weighted entropy after. A tree asks the question with the largest gain.

**Task.** Six chess pieces (king, queen, rook, bishop, knight, pawn), each equally likely. Score two yes-or-no questions by information gain.

**Inputs.** None.

**Formulas.** Information gain of a split into groups L and R: $H(\text{node}) - \frac{n_L}{n}H(L) - \frac{n_R}{n}H(R)$, where $n_L, n_R$ are the group sizes and $n = 6$.

- Question A splits the pieces 3 and 3.
- Question B sets the knight apart: 1 and 5.

**Define.** `H_root` (entropy of all six pieces), `gain_33` (gain of A), `gain_15` (gain of B). Bits, do not round.


In [ ]:
# no inputs


In [ ]:
# Exercise 2: define H_root, gain_33, gain_15
H_root = None
gain_33 = None
gain_15 = None


In [ ]:
check("H_root", H_root, "c07bd162bc5a184e")
check("gain_33", gain_33, "4c0fd2ebda038544")
check("gain_15", gain_15, "96a409843db2d83c")


In [ ]:
# Plot: entropy before and after each question. Gain = height of the first bar minus the second.
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
p_grid = np.linspace(0.001, 0.999, 200)
axes[0].plot(p_grid, -p_grid * np.log2(p_grid) - (1 - p_grid) * np.log2(1 - p_grid))
axes[0].set_xlabel("share of class 1, p"); axes[0].set_ylabel("entropy H (bits)"); axes[0].set_title("Two classes: H is largest at p = 1/2")
after = [H_root - gain_33, H_root - gain_15]
axes[1].bar(["before", "after A (3 / 3)", "after B (1 / 5)"], [H_root] + after, color=["C0", "C2", "C3"])
axes[1].set_ylabel("entropy (bits)"); axes[1].set_title(f"gains: A {gain_33:.2f}, B {gain_15:.2f}")
plt.tight_layout(); plt.show()


### Exercise 3: pruning strength chosen on validation data

**Task.** For each `alpha` in `alphas`, fit `DecisionTreeRegressor(random_state=0, ccp_alpha=alpha)` on `Xtr, ytr` and compute the validation RMSE on `Xva, yva` with `rmse`. Keep the alpha with the smallest validation RMSE (the first one if two tie).

**Inputs.** `Xtr, ytr, Xva, yva` (Saratoga homes, 70% training and 30% validation, `price` in US dollars), `alphas`, `rmse`.

**Define.** `best_val_rmse` (US dollars) and `best_leaves` (integer, leaves of the winning tree).

**Tolerance.** `best_val_rmse` within 0.5% and `best_leaves` within 3 of the right values (scikit-learn versions differ a little).


In [ ]:
homes = pd.read_csv(DATA / "SaratogaHouses.csv", index_col=0)
Xall = pd.get_dummies(homes.drop(columns="price"), drop_first=True).astype(float)
yall = homes.price.to_numpy()
order = np.random.default_rng(10).permutation(len(homes))
n_train = int(0.7 * len(homes))
Xtr, ytr = Xall.iloc[order[:n_train]], yall[order[:n_train]]
Xva, yva = Xall.iloc[order[n_train:]], yall[order[n_train:]]
alphas = DecisionTreeRegressor(random_state=0).cost_complexity_pruning_path(Xtr, ytr).ccp_alphas[::10]
print(len(Xtr), "training rows,", len(Xva), "validation rows,", len(alphas), "alphas")


In [ ]:
# Exercise 3: define best_val_rmse, best_leaves
best_val_rmse = None
best_leaves = None


In [ ]:
check_near("best_val_rmse", best_val_rmse, 60191.31, 0.005, relative=True)
check_near("best_leaves", best_leaves, 11, 3.5)


In [ ]:
# Plot: validation error against tree size. Too few leaves underfit, too many overfit.
sizes, errs = [], []
for alpha in alphas:
    t_ = DecisionTreeRegressor(random_state=0, ccp_alpha=alpha).fit(Xtr, ytr)
    sizes.append(t_.get_n_leaves()); errs.append(rmse(yva, t_.predict(Xva)))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(sizes, errs, "o-", ms=3)
ax.axvline(best_leaves, color="k", ls="--", label=f"best: {best_leaves} leaves")
ax.set_xscale("log"); ax.set_xlabel("leaves (log scale)"); ax.set_ylabel("validation RMSE (US dollars)"); ax.legend(frameon=False)
plt.show()


### Exercise 4: random forest and boosting scores

**Task.** Fit both models on `Xtr, ytr` from Exercise 3 and score them. Use exactly these settings and seeds.

- Forest: `RandomForestRegressor(n_estimators=200, max_features=1/3, min_samples_leaf=3, oob_score=True, random_state=0, n_jobs=-1)`. Its out-of-bag R² is the attribute `oob_score_`.
- Boosting: `GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0)`. Score it on `Xva, yva` with `rmse`.

**Define.** `oob_r2` (a number below 1, no unit) and `gb_val_rmse` (US dollars).

**Tolerance.** `oob_r2` within 0.02 and `gb_val_rmse` within 2% of the right values.


In [ ]:
# inputs: Xtr, ytr, Xva, yva, rmse from Exercise 3


In [ ]:
# Exercise 4: define oob_r2, gb_val_rmse
oob_r2 = None
gb_val_rmse = None


In [ ]:
check_near("oob_r2", oob_r2, 0.6525, 0.02)
check_near("gb_val_rmse", gb_val_rmse, 50751.4, 0.02, relative=True)


In [ ]:
# Plot: boosting adds trees one at a time. Validation error after each tree.
gb_plot = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0).fit(Xtr, ytr)
curve = [rmse(yva, p) for p in gb_plot.staged_predict(Xva)]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, 301), curve, label="boosting")
ax.axhline(best_val_rmse, color="C1", ls="--", label="best pruned single tree")
ax.set_xlabel("number of trees"); ax.set_ylabel("validation RMSE (US dollars)"); ax.legend(frameon=False)
plt.show()


### Exercise 5: why averaging trees has a floor

**Task.** Compute the mean squared error of an average of $B$ trees.

**Inputs.** `sigma2_tree` (mean squared error of one tree), `rho_bag` (correlation of two bagged trees' errors), `rho_forest` (the same for a random forest), `B_list`.

**Formula.** Average of $B$ trees: $\text{MSE}_B = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2$, where $\sigma^2$ is one tree's MSE and $\rho$ the correlation. As $B$ grows, $\text{MSE}_B \to \rho\sigma^2$.

**Define.**
- `mse_bag`: $\text{MSE}_B$ for bagging at each $B$ in `B_list`, shape (3,).
- `mse_forest`: $\text{MSE}_B$ for the forest at each $B$ in `B_list`, shape (3,).
- `floor_bag`, `floor_forest`: the limits $\rho\sigma^2$.


In [ ]:
sigma2_tree = 4.0
rho_bag, rho_forest = 0.6, 0.25
B_list = np.array([1, 10, 500])


In [ ]:
# Exercise 5: define mse_bag, mse_forest, floor_bag, floor_forest
mse_bag = None
mse_forest = None
floor_bag = None
floor_forest = None


In [ ]:
check_array("mse_bag", mse_bag, "82edb9aab7ac2dca", (3,))
check_array("mse_forest", mse_forest, "475841d062e108d0", (3,))
check("floor_bag", floor_bag, "69442248446573cc")
check("floor_forest", floor_forest, "032ef27b29be71c5")


In [ ]:
# Plot: MSE of an average of B trees. It falls to the floor rho * sigma^2, not to 0.
B_grid = np.arange(1, 201)
fig, ax = plt.subplots(figsize=(6, 3.5))
for rho_, name, col in [(rho_bag, "bagging", "C0"), (rho_forest, "random forest", "C1")]:
    ax.plot(B_grid, rho_ * sigma2_tree + (1 - rho_) / B_grid * sigma2_tree, color=col, label=f"{name}, rho = {rho_}")
    ax.axhline(rho_ * sigma2_tree, color=col, ls=":")
ax.set_xscale("log"); ax.set_xlabel("number of trees B (log scale)"); ax.set_ylabel("MSE of the average"); ax.legend(frameon=False)
plt.show()


## Part 2: Sparsity

Online lab: [Sparsity lab](https://vsokolov.org/labs/sparsity-lab.html).


### Exercise 6: lasso as a function of a z-score

**Task.** With uncorrelated genes, the lasso estimate is a function of one gene's z-score $z$. Apply it to the z-scores and count exact zeros.

**Inputs.** `z_genes` (z-scores of 8 genes), `lam_lasso`.

**Formulas.**
- Lasso (soft threshold): $\operatorname{sign}(z)\,\max(|z| - \lambda_{\text{lasso}},\, 0)$. Values with $|z| \le \lambda_{\text{lasso}}$ become exactly 0.
- Universal threshold for $p$ genes, $n$ patients, noise SD $\sigma$: $\lambda_{\text{univ}} = \sigma\sqrt{2\log p / n}$ (natural log).

**Define.**
- `z_lasso`: shape (8,).
- `zeros_lasso`: number of exact zeros (integer).
- `lam_univ`: $\lambda_{\text{univ}}$ for $p = 1000$, $n = 100$, $\sigma = 1$.


In [ ]:
z_genes = np.array([4.1, -0.3, 0.8, -2.6, 1.2, 0.1, -1.0, 3.0])
lam_lasso = 1.5


In [ ]:
# Exercise 6: define z_lasso, zeros_lasso, lam_univ
z_lasso = None
zeros_lasso = None
lam_univ = None


In [ ]:
check_array("z_lasso", z_lasso, "8555cd6d1606704f", (8,))
check("zeros_lasso", zeros_lasso, "5f8dd08868541644")
check("lam_univ", lam_univ, "1f6ff1e4357db548")


In [ ]:
# Plot: the soft threshold. Inside [-lambda, lambda] the estimate is exactly 0.
z_grid = np.linspace(-5, 5, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(z_grid, z_grid, color="grey", ls=":", label="no shrinkage")
ax.plot(z_grid, np.sign(z_grid) * np.maximum(np.abs(z_grid) - lam_lasso, 0), color="C0", label="lasso")
ax.axvspan(-lam_lasso, lam_lasso, color="C0", alpha=0.1)
ax.scatter(z_genes, z_lasso, color="C3", zorder=3, label="your 8 genes")
ax.set_xlabel("z-score"); ax.set_ylabel("estimate"); ax.legend(frameon=False)
plt.show()


### Exercise 7: needles in a haystack, p much larger than n

**Task.** 100 patients, 1,000 genes, 10 needles (true effect 1) and 990 hay genes (effect 0). Compare one-gene-at-a-time testing with the lasso.

**Inputs.** `X_g` (100 × 1000), `y_g` (100,), `needles` (indices of the 10 needles), `hay` (True for hay genes, shape (1000,)), `marginal_pvalues(X, y)` (p-value of each gene from its own simple regression), `Lasso` (scikit-learn).

**Steps.**
1. Screening: `pv = marginal_pvalues(X_g, y_g)`. Count hay genes with $p < 0.05$ (false alarms; expect about 5% of 990).
2. Lasso: `Lasso(alpha=lam_univ, max_iter=100_000, tol=1e-10).fit(X_g, y_g)`, with `lam_univ` from Exercise 6. A gene is selected when its coefficient in `.coef_` is not 0.

**Define.** `hay_sig` (hay genes with $p < 0.05$), `needles_found` (needles selected by the lasso), `hay_selected` (hay genes selected by the lasso). Integers.

**Tolerance.** Each count within 1 (scikit-learn versions differ a little).


In [ ]:
from scipy import stats
from sklearn.linear_model import Lasso

rng_g = np.random.default_rng(4)
X_g = rng_g.standard_normal((100, 1000))
needles = np.arange(10) * 100
beta_g = np.zeros(1000)
beta_g[needles] = 1.0
y_g = X_g @ beta_g + rng_g.standard_normal(100)
hay = np.ones(1000, bool)
hay[needles] = False

def marginal_pvalues(X, y):
    # two-sided p-value of the slope in y = a + b x_j, for each column j
    n = len(y)
    Xc, yc = X - X.mean(axis=0), y - y.mean()
    b = Xc.T @ yc / (Xc**2).sum(axis=0)
    resid = yc[:, None] - Xc * b
    se = np.sqrt((resid**2).sum(axis=0) / (n - 2) / (Xc**2).sum(axis=0))
    return 2 * stats.t.sf(np.abs(b / se), n - 2)

print(X_g.shape, "rank of X'X:", np.linalg.matrix_rank(X_g), "of 1000")


In [ ]:
# Exercise 7: define hay_sig, needles_found, hay_selected
hay_sig = None
needles_found = None
hay_selected = None


In [ ]:
check_near("hay_sig", hay_sig, 46, 1.01)
check_near("needles_found", needles_found, 10, 1.01)
check_near("hay_selected", hay_selected, 10, 1.01)


In [ ]:
# Plot: one gene at a time (left) against the lasso (right). Red: the 10 needles.
pv_plot = marginal_pvalues(X_g, y_g)
coef_plot = Lasso(alpha=lam_univ, max_iter=100_000, tol=1e-10).fit(X_g, y_g).coef_
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(np.where(hay)[0], -np.log10(pv_plot[hay]), s=6, color="grey", label="hay")
axes[0].scatter(needles, -np.log10(pv_plot[needles]), s=25, color="C3", label="needles")
axes[0].axhline(-np.log10(0.05), color="k", ls="--", label="p = 0.05")
axes[0].set_xlabel("gene"); axes[0].set_ylabel("-log10 p-value"); axes[0].set_title(f"{hay_sig} hay genes cross p = 0.05"); axes[0].legend(frameon=False, fontsize=8)
axes[1].vlines(np.where(hay)[0], 0, coef_plot[hay], color="grey")
axes[1].vlines(needles, 0, coef_plot[needles], color="C3", lw=2)
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_xlabel("gene"); axes[1].set_ylabel("lasso coefficient"); axes[1].set_title(f"lasso: {needles_found} needles, {hay_selected} hay")
plt.tight_layout(); plt.show()


## Part 3: Neural networks

Online lab: [Neural net lab](https://vsokolov.org/labs/neural-net-lab.html).


### Exercise 8: forward pass of a small network

**Task.** Run the forward pass of the network below for one new point and for all 200 circle points.

**Inputs.** `W1` (shape (2, 4)), `b1` (4,), `W2` (4,), `b2` (number), `x_new` (2,), `Xcirc` (200, 2), `ycirc` (200 labels, 0 for the center and 1 for the outer ring). There is no `relu` function: use `np.maximum(..., 0.0)`.

**Formula.** Hidden values $h = \max(x W_1 + b_1,\, 0)$ and output $o = h \cdot W_2 + b_2$. Predict class 1 when $o > 0$, otherwise class 0.

**Define.**
- `h_new`: hidden values for `x_new`, shape (4,).
- `y_new`: the output $o$ for `x_new`, a number.
- `H_all`: hidden values for all points, shape (200, 4).
- `acc_circle`: share of the 200 points whose predicted class equals `ycirc`, between 0 and 1.

Do not round.


In [ ]:
circle = pd.read_csv(DATA / "circle.csv")
Xcirc = circle[["x1", "x2"]].to_numpy()
ycirc = circle.label.to_numpy()
W1 = np.array([[1.0, -1.0, 0.0, 0.0],
               [0.0, 0.0, 1.0, -1.0]])
b1 = np.array([-4.0, -4.0, -4.0, -4.0])
W2 = np.array([1.0, 1.0, 1.0, 1.0])
b2 = -0.5
x_new = np.array([-6.0, 5.5])
print(Xcirc.shape, ycirc.shape)


In [ ]:
# Exercise 8: define h_new, y_new, H_all, acc_circle
h_new = None
y_new = None
H_all = None
acc_circle = None


In [ ]:
check_array("h_new", h_new, "4771e4785f1e4c5c", (4,))
check("y_new", y_new, "a98b80685db68ac1")
check_array("H_all", H_all, "9512dd8cd2b7e589", (200, 4))
check("acc_circle", acc_circle, "bbaa670b89efc325")


In [ ]:
# Plot: each hidden unit is one line. The network says "outer ring" when any unit switches on.
fig, ax = plt.subplots(figsize=(4.5, 4.5))
pred = ((H_all @ W2 + b2) > 0).astype(int)
ax.scatter(Xcirc[:, 0], Xcirc[:, 1], c=pred, cmap="coolwarm", s=12)
for v in (4, -4):
    ax.axvline(v, color="k", lw=0.8); ax.axhline(v, color="k", lw=0.8)
ax.scatter(*x_new, marker="*", s=200, color="gold", edgecolor="k", label="x_new")
ax.set_aspect("equal"); ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_title(f"accuracy {acc_circle:.2f}"); ax.legend(frameon=False)
plt.show()


### Exercise 9: three ReLU folds

**Task.** Evaluate a one-layer ReLU network on two inputs and count its flat pieces.

**Inputs.** `W_fold` (3 × 2), `b_fold` (3,), `v_fold` (3,), `c_fold` (number), `pts` (4 points, shape (4, 2)).

**Formula.** $F(x) = c + \sum_{k=1}^{3} v_k \max(w_k \cdot x + b_k,\, 0)$, where $w_k$ is row $k$ of `W_fold`. Unit $k$ is on when $w_k \cdot x + b_k > 0$. Each unit folds the plane along the line $w_k \cdot x + b_k = 0$. $n$ lines in general position cut the plane into at most $1 + n + \binom{n}{2}$ regions.

**Define.**
- `F_pts`: $F$ at the 4 points, shape (4,).
- `units_on`: number of units on at each point, shape (4,).
- `max_pieces_3`, `max_pieces_6`: the region bound for $n = 3$ and $n = 6$.


In [ ]:
W_fold = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
b_fold = np.array([-2.0, -2.0, -3.0])
v_fold = np.array([1.0, 1.0, -1.0])
c_fold = 1.0
pts = np.array([[0.0, 0.0], [2.5, 0.0], [1.0, 2.5], [3.0, 3.0]])


In [ ]:
# Exercise 9: define F_pts, units_on, max_pieces_3, max_pieces_6
F_pts = None
units_on = None
max_pieces_3 = None
max_pieces_6 = None


In [ ]:
check_array("F_pts", F_pts, "48ee0ea25b904173", (4,))
check_array("units_on", units_on, "c2a486a1fdc56515", (4,))
check("max_pieces_3", max_pieces_3, "8aa06b7a831cc628")
check("max_pieces_6", max_pieces_6, "8040d29ce9086e01")


In [ ]:
# Plot: F over the plane. Dashed lines are the three folds. In each region F is a plane; in the top-right corner it is flat.
g1, g2 = np.meshgrid(np.linspace(-1, 4.5, 300), np.linspace(-1, 4.5, 300))
G = np.column_stack([g1.ravel(), g2.ravel()])
F_grid = (c_fold + np.maximum(G @ W_fold.T + b_fold, 0.0) @ v_fold).reshape(g1.shape)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(np.round(F_grid, 9), origin="lower", extent=(-1, 4.5, -1, 4.5), cmap="viridis"); fig.colorbar(im, label="F")
xs = np.linspace(-1, 4.5, 2)
ax.axvline(2, color="w", ls="--"); ax.axhline(2, color="w", ls="--"); ax.plot(xs, 3 - xs, color="w", ls="--")
ax.scatter(pts[:, 0], pts[:, 1], color="r", zorder=3)
for (a, b), f, u in zip(pts, F_pts, units_on):
    ax.annotate(f"F={f:g}, on={u}", (a, b), color="w", xytext=(4, 4), textcoords="offset points")
ax.set_xlim(-1, 4.5); ax.set_ylim(-1, 4.5); ax.set_xlabel("x1"); ax.set_ylabel("x2")
plt.show()


### Exercise 10: softmax, cross-entropy and its gradient

**Task.** For 4 examples and 3 classes, compute the probabilities, the mean loss and the gradient of the mean loss with respect to the scores. Work along the class axis (`axis=1`) and subtract each row's maximum before `np.exp`.

**Inputs.** `S_ex` (shape (4, 3), raw scores) and `lab_ex` (4 labels, each 0, 1 or 2).

**Formula.** $p_{ik} = e^{s_{ik}} / \sum_j e^{s_{ij}}$. Loss $= -\frac{1}{n}\sum_i \log p_{i,c_i}$, where $c_i$ is the true class of example $i$ and $n = 4$. Gradient $= (P - Y)/n$, where $Y$ is the one-hot matrix `np.eye(3)[lab_ex]`.

**Define.** `P_ex` (shape (4, 3)), `loss_ex` (a number, in nats) and `grad_scores` (shape (4, 3)). Do not round.


In [ ]:
S_ex = np.array([[2.0, 0.5, -1.0], [0.0, 0.0, 0.0], [1.0, 3.0, 0.0], [-1.0, 0.0, 2.0]])
lab_ex = np.array([0, 2, 1, 2])


In [ ]:
# Exercise 10: define P_ex, loss_ex, grad_scores
P_ex = None
loss_ex = None
grad_scores = None


In [ ]:
check_array("P_ex", P_ex, "b95c673a8f668257", (4, 3))
check("loss_ex", loss_ex, "1ade89f06c21799a")
check_array("grad_scores", grad_scores, "e31f99c3062f1e7a", (4, 3))


In [ ]:
# Plot: softmax probabilities of each example. Black outline: the true class.
fig, axes = plt.subplots(1, 4, figsize=(10, 2.6), sharey=True)
for i, ax in enumerate(axes):
    bars = ax.bar([0, 1, 2], P_ex[i], color="C0")
    bars[lab_ex[i]].set_edgecolor("k"); bars[lab_ex[i]].set_linewidth(2.5)
    ax.set_title(f"example {i}, true {lab_ex[i]}"); ax.set_xticks([0, 1, 2]); ax.set_ylim(0, 1)
axes[0].set_ylabel("probability")
plt.tight_layout(); plt.show()


## Part 4: Gradient descent and SGD

Online lab: [SGD lab](https://vsokolov.org/labs/sgd-lab.html).


### Exercise 11: gradient descent steps

**Task.** Fit the one-weight logistic model $p_i = \sigma(w x_i)$ by gradient descent. First do one step with the formula, then five steps in a loop.

**Inputs.** `x_ex`, `y_ex` (arrays of length 4), `w_start_ex` (starting weight), `eta_ex` (learning rate), `sigmoid`.

**Formulas.** Gradient of the mean log loss: $g(w) = \frac{1}{4}\sum_i (p_i - y_i)\,x_i$. Update: $w \leftarrow w - \eta\, g(w)$. Mean log loss: $L(w) = -\frac{1}{4}\sum_i [\,y_i \log p_i + (1-y_i)\log(1-p_i)\,]$.

**Define.**
- `grad_ex`: $g$ at `w_start_ex`.
- `w_new`: the weight after one step from `w_start_ex`.
- `w_after5`: the weight after five steps in total, starting from `w_start_ex`.
- `loss_after5`: $L$ at `w_after5`.

Do not round.


In [ ]:
x_ex = np.array([1.0, -2.0, 0.5, 3.0])
y_ex = np.array([1.0, 0.0, 0.0, 1.0])
w_start_ex, eta_ex = 0.5, 0.4


In [ ]:
# Exercise 11: define grad_ex, w_new, w_after5, loss_after5
grad_ex = None
w_new = None
w_after5 = None
loss_after5 = None


In [ ]:
check("grad_ex", grad_ex, "37e8416a26ab696f")
check("w_new", w_new, "7f7d619c6c8a4148")
check("w_after5", w_after5, "8e44ebffe61677dc")
check("loss_after5", loss_after5, "d1f9a53b46a98912")


In [ ]:
# Plot: the loss curve and the five gradient descent steps.
w_grid = np.linspace(-1, 2.5, 200)
def mean_loss(w):
    p_ = sigmoid(w * x_ex)
    return -np.mean(y_ex * np.log(p_) + (1 - y_ex) * np.log(1 - p_))
ws = [w_start_ex]
for _ in range(5):
    ws.append(ws[-1] - eta_ex * np.mean((sigmoid(ws[-1] * x_ex) - y_ex) * x_ex))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(w_grid, [mean_loss(w) for w in w_grid])
ax.plot(ws, [mean_loss(w) for w in ws], "o-", color="C3", label="steps")
ax.set_xlabel("w"); ax.set_ylabel("mean log loss L(w)"); ax.legend(frameon=False)
plt.show()


### Exercise 12: the step-size ceiling

**Task.** For the line $\theta_1 + \theta_2 x$ with loss $L(\theta) = \frac{1}{2n}\sum_i (y_i - \theta_1 - \theta_2 x_i)^2$, find the largest step size for which gradient descent converges, with $x$ centred at 0 and with $x$ shifted to mean 3.

**Inputs.** `x_c` (200 inputs with mean 0), `x_s = x_c + 3`.

**Formulas.** Design matrix $X$ with columns $(1, x_i)$. Hessian $H = X^\top X / n$, with eigenvalues $\lambda_{\min} \le \lambda_{\max}$ (`np.linalg.eigvalsh`). Gradient descent converges only when $\eta < 2/\lambda_{\max}$. Condition number $\kappa = \lambda_{\max}/\lambda_{\min}$.

**Define.** `eta_max_c`, `eta_max_s` ($2/\lambda_{\max}$ for each $x$), `kappa_c`, `kappa_s`. Do not round.


In [ ]:
rng_s = np.random.default_rng(1)
x_c = rng_s.standard_normal(200)
x_c = x_c - x_c.mean()
x_s = x_c + 3


In [ ]:
# Exercise 12: define eta_max_c, eta_max_s, kappa_c, kappa_s
eta_max_c = None
eta_max_s = None
kappa_c = None
kappa_s = None


In [ ]:
check("eta_max_c", eta_max_c, "033b64f1fa179555")
check("eta_max_s", eta_max_s, "e276c509547fbdea")
check("kappa_c", kappa_c, "e479e6c4071cc4a0")
check("kappa_s", kappa_s, "ab86fa6ad3d9b6d3")


In [ ]:
# Plot: loss contours and 30 gradient descent steps at half the ceiling. Star: the minimum. Shifted x: a long narrow valley, slow progress.
y_line = 1 + 2 * x_c + np.random.default_rng(3).standard_normal(200)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, x_, eta_, name in [(axes[0], x_c, eta_max_c, "x centred"), (axes[1], x_s, eta_max_s, "x mean 3")]:
    Xd = np.column_stack([np.ones_like(x_), x_])
    th_hat = np.linalg.lstsq(Xd, y_line, rcond=None)[0]
    t1, t2 = np.meshgrid(np.linspace(th_hat[0] - 4, th_hat[0] + 4, 150), np.linspace(th_hat[1] - 2, th_hat[1] + 2, 150))
    R = y_line[None, None, :] - t1[..., None] - t2[..., None] * x_
    ax.contour(t1, t2, (R**2).mean(axis=-1) / 2, levels=25, cmap="Greys")
    th, path = np.array([th_hat[0] - 3.5, th_hat[1] + 1.5]), []
    for _ in range(30):
        path.append(th.copy()); th = th - 0.5 * eta_ * Xd.T @ (Xd @ th - y_line) / len(x_)
    path = np.array(path)
    ax.plot(path[:, 0], path[:, 1], "o-", ms=3, color="C3")
    ax.scatter(*th_hat, marker="*", s=250, color="gold", edgecolor="k", zorder=3)
    ax.set_title(f"{name}: eta = {0.5 * eta_:.3f}, 30 steps"); ax.set_xlabel("theta1 (intercept)"); ax.set_ylabel("theta2 (slope)")
plt.tight_layout(); plt.show()


### Exercise 13: constant steps hover, shrinking steps converge

**Task.** Estimate a mean $\theta$ by SGD with batch size 1. The loss of one example is $\frac12(\theta - y_i)^2$, so its gradient is $\theta - y_i$. Run 5,000 steps from $\theta_0 = 0$, using example `idx_sgd[t]` at step $t$, three ways.

**Inputs.** `y_sgd` (200 values), `idx_sgd` (5,000 example indices), `eta0`.

**Formulas.** Step $t = 0, 1, \ldots, 4999$: $\theta_{t+1} = \theta_t - \eta_t\,(\theta_t - y_{\text{idx}[t]})$.
- Constant: $\eta_t = \eta_0$.
- Shrinking: $\eta_t = \eta_0 / (1 + t/100)$.
- Polyak average: the mean of the 5,000 iterates $\theta_1, \ldots, \theta_{5000}$ of the constant run.

The exact answer is $\bar y$, the mean of `y_sgd`.

**Define.** `err_const`, `err_shrink`, `err_polyak`: $|\theta - \bar y|$ for the last constant iterate, the last shrinking iterate and the Polyak average. Do not round.


In [ ]:
rng_m = np.random.default_rng(2)
y_sgd = 1 + rng_m.standard_normal(200)
idx_sgd = rng_m.integers(0, 200, size=5000)
eta0 = 0.1


In [ ]:
# Exercise 13: define err_const, err_shrink, err_polyak
err_const = None
err_shrink = None
err_polyak = None


In [ ]:
check("err_const", err_const, "d19aa94465629a7c")
check("err_shrink", err_shrink, "abae713b46cdf6d6")
check("err_polyak", err_polyak, "8cd519f83c074d3b")


In [ ]:
# Plot: the three estimates over 5,000 steps. The constant step keeps bouncing around the answer.
th_c, th_d, run_c, run_d = 0.0, 0.0, [], []
for t in range(5000):
    yi = y_sgd[idx_sgd[t]]
    th_c -= eta0 * (th_c - yi); th_d -= eta0 / (1 + t / 100) * (th_d - yi)
    run_c.append(th_c); run_d.append(th_d)
steps = np.arange(1, 5001)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(steps, run_c, lw=0.6, alpha=0.7, label="constant step")
ax.plot(steps, run_d, label="shrinking step")
ax.plot(steps, np.cumsum(run_c) / steps, label="Polyak average")
ax.axhline(y_sgd.mean(), color="k", ls="--", label="exact answer")
ax.set_ylim(y_sgd.mean() - 1, y_sgd.mean() + 1); ax.set_xlabel("step"); ax.set_ylabel("theta"); ax.legend(frameon=False, ncol=2)
plt.show()


### Exercise 14: minibatch SGD in PyTorch

**Task.** Train a network on the simulated data with plain SGD in minibatches of 64, for 5 epochs, with the fixed seeds already in the cell. After each epoch record the validation loss. Keep a copy of the weights with the lowest validation loss and restore them at the end.

**Inputs.** `make_loader(0)` (training batches), `make_model(0)` (784 to 32 to 10 network), `evaluate(model, loader)` (returns average cross-entropy and accuracy), `val_loader`, learning rate 0.2.

**Define.** `val_loss` (cross-entropy in nats) and `val_acc` (between 0 and 1) of the restored model on the validation set.

**Tolerance.** `val_loss` within 0.1 and `val_acc` within 0.06 of the right values (other seeds move the result by up to 0.08 and 0.05). The check also tests `best_epoch` and the restore.


In [ ]:
# SIMULATED data: 10 classes, 784 features. Each class has a random centre; an example is its centre (scaled) plus noise.
def synthetic_split(n_train=2000, n_val=1000):
    r = np.random.default_rng(0)
    centres = r.normal(size=(10, 784))
    def draw(n):
        y = r.integers(0, 10, size=n)
        x = 0.1 * centres[y] + r.normal(size=(n, 784))
        return TensorDataset(torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return draw(n_train), draw(n_val)

train_set, val_set = synthetic_split()
val_loader = DataLoader(val_set, batch_size=1000)

def make_loader(seed=0):
    return DataLoader(train_set, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed))

def make_model(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(784, 32), nn.ReLU(), nn.Linear(32, 10))

def evaluate(model, loader):
    # average cross-entropy and accuracy, without changing the weights
    model.eval()
    with torch.no_grad():
        xb_, yb_ = next(iter(loader))
        out = model(xb_)
        result = nn.functional.cross_entropy(out, yb_).item(), (out.argmax(dim=1) == yb_).float().mean().item()
    model.train()
    return result
print(len(train_set), "training examples,", len(val_set), "validation examples (simulated data)")


In [ ]:
# Exercise 14: fill in each line marked FILL, then define val_loss and val_acc
train_loader = make_loader(0)
model = make_model(0)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.2)
val_curve, best_val, best_epoch, best_state = [], float("inf"), None, None
for epoch in range(5):
    for xb, yb in train_loader:
        scores = None   # FILL: forward pass (raw scores)
        loss = None     # FILL: loss from scores and yb
        # FILL: zero the gradients
        # FILL: backward pass
        # FILL: SGD step
    val_curve.append(evaluate(model, val_loader)[0])
    # FILL: if this epoch has the lowest validation loss so far, set best_val, best_epoch (from 0) and best_state (a copy.deepcopy of model.state_dict())
# FILL: load best_state into model
val_loss = None   # FILL: validation loss of the restored model
val_acc = None    # FILL: validation accuracy of the restored model


In [ ]:
assert best_epoch is not None and best_epoch == int(np.argmin(val_curve)), "not yet: best_epoch must be the epoch with the lowest validation loss"
assert abs(evaluate(model, val_loader)[0] - min(val_curve)) < 1e-5, "not yet: restore the best weights into model"
check_near("val_loss", val_loss, 0.9202, 0.1)
check_near("val_acc", val_acc, 0.6990, 0.06)


In [ ]:
# Plot: validation loss after each epoch. The star marks the weights you restored.
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(len(val_curve)), val_curve, "o-")
ax.scatter([best_epoch], [val_curve[best_epoch]], marker="*", s=250, color="gold", edgecolor="k", zorder=3)
ax.set_xlabel("epoch"); ax.set_ylabel("validation loss (nats)")
plt.show()


## Part 5: A network that plays chess

Online lab: [Chess lab](https://vsokolov.org/labs/chess-lab.html).


### Exercise 15: the chess network, value and search

**Task.** Three computations from the chess lab's network.

**Inputs.** `layer_sizes` (widths from the 772 board inputs through the ten ReLU layers), `n_moves` (4,096 policy outputs), `cp` (engine evaluation in centipawns), `Q_cand`, `P_cand`, `N_cand` (three candidate moves), `c_puct`.

**Formulas.**
- A fully connected layer from $a$ to $b$ units has $ab + b$ parameters (weights and biases). The trunk is the ten layers in `layer_sizes`. Two heads sit on the last layer of 128: policy (128 to 4,096) and value (128 to 1).
- Value from centipawns: $v = 1/(1 + 10^{-\text{cp}/400})$, the chance that the side to move wins.
- Search score of move $a$: $Q(a) + c\,P(a)\,\dfrac{\sqrt{N}}{1 + N(a)}$, where $Q$ is the average value found so far, $P$ the policy prior, $N(a)$ the visits to move $a$, and $N = \sum_a N(a)$.

**Define.**
- `n_params`: total parameters, trunk plus both heads (integer).
- `v_cp`: $v$ for `cp`.
- `scores_cand`: search scores of the 3 moves, shape (3,).
- `move_next`: index (0, 1 or 2) of the move the search visits next, the largest score.


In [ ]:
layer_sizes = [772, 256, 256, 256, 256, 256, 256, 256, 256, 256, 128]
n_moves = 4096
cp = 150
Q_cand = np.array([0.62, 0.55, 0.40])
P_cand = np.array([0.30, 0.15, 0.50])
N_cand = np.array([40, 6, 2])
c_puct = 1.5


In [ ]:
# Exercise 15: define n_params, v_cp, scores_cand, move_next
n_params = None
v_cp = None
scores_cand = None
move_next = None


In [ ]:
check("n_params", n_params, "1240539e9b397a28")
check("v_cp", v_cp, "9b825d2ea3320e3e")
check_array("scores_cand", scores_cand, "c2116ec4f8814f00", (3,))
check("move_next", move_next, "8df4dcdd9ea3b4a3")


In [ ]:
# Plot: search score = Q (exploit) + bonus (explore). Few visits and a high prior give a big bonus.
bonus = scores_cand - Q_cand
fig, ax = plt.subplots(figsize=(6, 3.5))
names = [f"move {i}\nN = {n}, P = {p}" for i, (n, p) in enumerate(zip(N_cand, P_cand))]
ax.bar(names, Q_cand, label="Q, value so far")
ax.bar(names, bonus, bottom=Q_cand, label="exploration bonus")
ax.set_ylabel("search score"); ax.set_title(f"next visit: move {move_next}"); ax.legend(frameon=False)
plt.show()
